# EXP14: 趋势 EMA 周期扫描

EMA100 是趋势过滤的核心。测试 EMA [50,75,100,125,150,200] 对信号质量的影响。

In [ ]:
import sys; sys.path.insert(0, '..')
from backtest import DataBundle, run_variant, run_kfold, C12_KWARGS
from backtest.stats import print_ranked

LIVE_KWARGS = {
    **C12_KWARGS,
    "intraday_adaptive": True,
    "choppy_threshold": 0.35,
    "kc_only_threshold": 0.60,
    "spread_cost": 0.50,
}
print('Ready')

In [ ]:
results = []
for ema in [50, 75, 100, 125, 150, 200]:
    data = DataBundle.load_custom(ema_trend=ema)
    r = run_variant(data, f"EMA_trend={ema}", **LIVE_KWARGS)
    r['ema_trend'] = ema
    results.append(r)
    print(f"  EMA{ema}: Sharpe={r['sharpe']:.2f}, N={r['n']}, PnL=${r['total_pnl']:.0f}")

print_ranked(results)

In [ ]:
# K-Fold on champion
best = sorted(results, key=lambda r: r['sharpe'], reverse=True)[0]
BEST_EMA = best['ema_trend']
print(f"Champion: EMA_trend={BEST_EMA}, Sharpe={best['sharpe']:.2f}")

if BEST_EMA != 100:
    data_champ = DataBundle.load_custom(ema_trend=BEST_EMA)
    data_base = DataBundle.load_default()
    champ_folds = run_kfold(data_champ, LIVE_KWARGS, label_prefix="Champ_")
    base_folds = run_kfold(data_base, LIVE_KWARGS, label_prefix="Base_")
    for b, c in zip(base_folds, champ_folds):
        print(f"{b['fold']}: Base={b['sharpe']:.2f}  EMA{BEST_EMA}={c['sharpe']:.2f}  D={c['sharpe']-b['sharpe']:+.2f}")
else:
    print("EMA100 is already optimal, no K-Fold needed")

In [ ]:
import json
from backtest.runner import sanitize_for_json
with open('../data/exp14_results.json', 'w') as f:
    json.dump(sanitize_for_json(results), f, indent=2)
print('Saved')